# WWF Data Ingestion Pipeline - Vector Store Creation

This notebook creates a comprehensive vector store from WWF PDF documents for:
- Micro-learning module generation
- Chatbot RAG system
- Multi-category knowledge base

**Key Features:**
- Intelligent PDF chunking with 800 token overlap
- Category-based metadata filtering
- Persistent ChromaDB storage
- Quality validation

In [1]:


import os
import sys
from pathlib import Path
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Document processing
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector store and embeddings
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions

# Utilities
import json
from datetime import datetime
import time
from dotenv import load_dotenv

# Load environment variables
env_path = Path(r"C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\.env")
load_dotenv(env_path)

print("✅ All libraries imported successfully")
print(f"⏰ Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ All libraries imported successfully
⏰ Started at: 2026-04-07 19:10:44


## Configuration and Setup

In [2]:

import csv

# Configuration
BASE_DIR = Path(r"C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF")
DATA_DIR = BASE_DIR / "data"
VECTOR_STORE_PATH = BASE_DIR / "vector_store"
CATEGORY_FILE_PATH = DATA_DIR / "category_file" / "categories.csv"

# Create vector store directory
VECTOR_STORE_PATH.mkdir(exist_ok=True)

# Chunking configuration
CHUNK_SIZE = 1000  # Characters per chunk
CHUNK_OVERLAP = 300  # High overlap for better context preservation
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

# OpenAI Embedding Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
EMBEDDING_MODEL = "text-embedding-3-small"  # OpenAI's cheapest: $0.02 per 1M tokens

if not OPENAI_API_KEY:
    raise ValueError("❌ OPENAI_API_KEY not found in environment variables")

# Load categories dynamically from CSV (same as main app)
def normalize_category_name(category_name: str) -> str:
    """Convert category display name to folder name format."""
    return category_name.lower().replace(' & ', '_and_').replace(' ', '_')

def load_categories_from_csv():
    """Load active categories from CSV file and build course_id mappings."""
    categories = set()
    course_id_to_category = {}
    category_to_course_id = {}
    
    if not CATEGORY_FILE_PATH.exists():
        raise FileNotFoundError(f"categories.csv not found at {CATEGORY_FILE_PATH}")
    
    with open(CATEGORY_FILE_PATH, 'r', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            if row.get('active', '').lower() == 'true':
                category_display_name = row.get('category_name', '').strip()
                course_id = row.get('course_id', '').strip()
                
                if category_display_name and course_id:
                    category_folder_name = normalize_category_name(category_display_name)
                    categories.add(category_folder_name)
                    
                    if course_id not in course_id_to_category:
                        course_id_to_category[course_id] = category_folder_name
                    
                    # Create reverse mapping (category -> course_id)
                    if category_folder_name not in category_to_course_id:
                        category_to_course_id[category_folder_name] = course_id
    
    return sorted(list(categories)), course_id_to_category, category_to_course_id

# Load categories from CSV
CATEGORIES, COURSE_ID_TO_CATEGORY, CATEGORY_TO_COURSE_ID = load_categories_from_csv()

print(f"📁 Base Directory: {BASE_DIR}")
print(f"📁 Data Directory: {DATA_DIR}")
print(f"📁 Category CSV: {CATEGORY_FILE_PATH}")
print(f"🤖 Embedding Model: {EMBEDDING_MODEL} (OpenAI)")
print(f"💰 Cost: $0.02 per 1M tokens")
print(f"📐 Dimensions: 1536")
print(f"💾 Vector Store Path: {VECTOR_STORE_PATH}")
print(f"🎯 Categories loaded from CSV: {len(CATEGORIES)}")
print(f"📋 Course ID → Category: {COURSE_ID_TO_CATEGORY}")
print(f"📋 Category → Course ID: {CATEGORY_TO_COURSE_ID}")
print(f"📂 Categories: {CATEGORIES}")
print(f"📊 Chunk Size: {CHUNK_SIZE} | Overlap: {CHUNK_OVERLAP}")

📁 Base Directory: C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF
📁 Data Directory: C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\data
📁 Category CSV: C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\data\category_file\categories.csv
🤖 Embedding Model: text-embedding-3-small (OpenAI)
💰 Cost: $0.02 per 1M tokens
📐 Dimensions: 1536
💾 Vector Store Path: C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\vector_store
🎯 Categories loaded from CSV: 3
📋 Course ID → Category: {'003': 'sustainable_agriculture_and_natural_resources', '002': 'sustainability_strategy_and_compliance', '001': 'circular_economy_and_waste_reduction'}
📋 Category → Course ID: {'sustainable_agriculture_and_natural_resources': '003', 'sustainability_strategy_and_compliance': '002', 'circular_economy_and_waste_reduction': '001'}
📂 Categories: ['circular_economy_and_waste_reduction', 'sustainability_strategy_and

## Step 1: Initialize Embedding Model and Vector Store

In [3]:
# Initialize OpenAI Embedding Function for ChromaDB

print("🔄 Initializing OpenAI embeddings...")
print(f"🤖 Model: {EMBEDDING_MODEL}")
print(f"📐 Dimensions: 1536")
print(f"💰 Cost: $0.02 per 1M tokens (very cheap!)")

# Create OpenAI embedding function
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=EMBEDDING_MODEL
)

print(f"✅ OpenAI embedding function initialized")

# Initialize ChromaDB with persistent storage
print("\n🔄 Initializing ChromaDB...")
chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_PATH),
    settings=Settings(
        anonymized_telemetry=False,
        allow_reset=True
    )
)

# Create or get collection
COLLECTION_NAME = "wwf_knowledge_base"

try:
    # Delete existing collection if it exists (for fresh start)
    try:
        chroma_client.delete_collection(name=COLLECTION_NAME)
        print(f"🗑️  Deleted existing collection: {COLLECTION_NAME}")
    except:
        pass
    
    # Create new collection with OpenAI embeddings
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        embedding_function=openai_ef,
        metadata={"description": "WWF Multi-Category Knowledge Base for RAG with OpenAI Embeddings"}
    )
    print(f"✅ Created new collection: {COLLECTION_NAME}")
    print(f"   Embeddings will be generated via OpenAI API")
except Exception as e:
    print(f"⚠️  Using existing collection")
    collection = chroma_client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=openai_ef
    )
    print(f"❌ Error creating collection: {e}")

🔄 Initializing OpenAI embeddings...
🤖 Model: text-embedding-3-small
📐 Dimensions: 1536
💰 Cost: $0.02 per 1M tokens (very cheap!)
✅ OpenAI embedding function initialized

🔄 Initializing ChromaDB...
🗑️  Deleted existing collection: wwf_knowledge_base
✅ Created new collection: wwf_knowledge_base
   Embeddings will be generated via OpenAI API


## Step 2: PDF Extraction and Chunking Functions

In [4]:


def extract_text_from_pdf(pdf_path: Path) -> Tuple[str, int]:
    """
    Extract text from a PDF file.
    
    Args:
        pdf_path: Path to the PDF file
        
    Returns:
        Tuple of (extracted_text, page_count)
    """
    try:
        reader = PdfReader(str(pdf_path))
        text = ""
        for page_num, page in enumerate(reader.pages, 1):
            page_text = page.extract_text()
            if page_text:
                text += f"\n--- Page {page_num} ---\n{page_text}"
        
        return text, len(reader.pages)
    
    except Exception as e:
        print(f"❌ Error extracting from {pdf_path.name}: {e}")
        return "", 0


def chunk_text_intelligently(text: str, source_file: str, category: str) -> List[Dict]:
    """
    Chunk text with proper overlap and metadata.
    
    Args:
        text: Text to chunk
        source_file: Name of source PDF
        category: Category name
        
    Returns:
        List of chunk dictionaries with text and metadata
    """
    # Get course_id from category
    course_id = CATEGORY_TO_COURSE_ID.get(category, "unknown")
    
    # Initialize text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=SEPARATORS,
        is_separator_regex=False
    )
    
    # Split text
    chunks = text_splitter.split_text(text)
    
    # Create chunk dictionaries with metadata
    chunk_dicts = []
    for idx, chunk in enumerate(chunks, 1):
        chunk_dict = {
            "text": chunk.strip(),
            "metadata": {
                "category": category,
                "course_id": course_id,  # Added for filtering by CourseID in API/chatbot
                "source_file": source_file,
                "chunk_index": idx,
                "total_chunks": len(chunks),
                "doc_type": "pdf",
                "ingestion_date": datetime.now().isoformat()
            }
        }
        chunk_dicts.append(chunk_dict)
    
    return chunk_dicts


print("✅ PDF processing functions defined")
print(f"📋 Metadata will include: category, course_id, source_file, chunk_index, total_chunks, doc_type, ingestion_date")

✅ PDF processing functions defined
📋 Metadata will include: category, course_id, source_file, chunk_index, total_chunks, doc_type, ingestion_date


## Step 3: Process All PDFs and Create Vector Store

In [5]:

def process_category(category: str) -> Dict:
    """
    Process all PDFs in a category directory.
    
    Args:
        category: Category name
        
    Returns:
        Statistics dictionary
    """
    category_dir = DATA_DIR / category
    
    if not category_dir.exists():
        print(f"⚠️  Category directory not found: {category}")
        return {"category": category, "pdfs": 0, "chunks": 0, "errors": []}
    
    # Find all PDFs
    pdf_files = list(category_dir.glob("*.pdf"))
    
    if not pdf_files:
        print(f"⚠️  No PDFs found in: {category}")
        return {"category": category, "pdfs": 0, "chunks": 0, "errors": []}
    
    print(f"\n{'='*80}")
    print(f"📂 Processing Category: {category}")
    print(f"📄 Found {len(pdf_files)} PDF(s)")
    print(f"{'='*80}")
    
    all_chunks = []
    stats = {
        "category": category,
        "pdfs": len(pdf_files),
        "chunks": 0,
        "errors": []
    }
    
    for pdf_idx, pdf_path in enumerate(pdf_files, 1):
        print(f"\n📄 [{pdf_idx}/{len(pdf_files)}] Processing: {pdf_path.name}")
        
        # Extract text
        text, page_count = extract_text_from_pdf(pdf_path)
        
        if not text:
            error_msg = f"Failed to extract text from {pdf_path.name}"
            stats["errors"].append(error_msg)
            print(f"   ❌ {error_msg}")
            continue
        
        print(f"   ✅ Extracted {len(text):,} characters from {page_count} pages")
        
        # Chunk text
        chunks = chunk_text_intelligently(text, pdf_path.name, category)
        all_chunks.extend(chunks)
        stats["chunks"] += len(chunks)
        
        print(f"   ✅ Created {len(chunks)} chunks with {CHUNK_OVERLAP} character overlap")
    
    return stats, all_chunks


# Process all categories
print("\n" + "="*80)
print("🚀 STARTING DATA INGESTION PIPELINE")
print("="*80)

all_statistics = []
total_chunks_to_add = []

for category in CATEGORIES:
    stats, chunks = process_category(category)
    all_statistics.append(stats)
    total_chunks_to_add.extend(chunks)

print("\n" + "="*80)
print(f"📊 INGESTION SUMMARY")
print("="*80)
for stat in all_statistics:
    print(f"\n📂 {stat['category']}")
    print(f"   PDFs processed: {stat['pdfs']}")
    print(f"   Chunks created: {stat['chunks']}")
    if stat['errors']:
        print(f"   ⚠️  Errors: {len(stat['errors'])}")
        for error in stat['errors']:
            print(f"      - {error}")

print(f"\n{'='*80}")
print(f"💾 Total chunks to store: {len(total_chunks_to_add):,}")
print(f"{'='*80}")


🚀 STARTING DATA INGESTION PIPELINE

📂 Processing Category: circular_economy_and_waste_reduction
📄 Found 2 PDF(s)

📄 [1/2] Processing: cewr_1.pdf
   ✅ Extracted 48,310 characters from 14 pages
   ✅ Created 70 chunks with 300 character overlap

📄 [2/2] Processing: cewr_2.pdf
   ✅ Extracted 71,631 characters from 31 pages
   ✅ Created 103 chunks with 300 character overlap

📂 Processing Category: sustainability_strategy_and_compliance
📄 Found 1 PDF(s)

📄 [1/1] Processing: ssc_1.pdf
   ✅ Extracted 58,483 characters from 21 pages
   ✅ Created 86 chunks with 300 character overlap

📂 Processing Category: sustainable_agriculture_and_natural_resources
📄 Found 2 PDF(s)

📄 [1/2] Processing: agri_nr_1.pdf
   ✅ Extracted 15,710 characters from 11 pages
   ✅ Created 23 chunks with 300 character overlap

📄 [2/2] Processing: agri_nr_2.pdf
   ✅ Extracted 21,328 characters from 16 pages
   ✅ Created 30 chunks with 300 character overlap

📊 INGESTION SUMMARY

📂 circular_economy_and_waste_reduction
   PDFs

## Step 4: Generate Embeddings and Store in ChromaDB

In [7]:
print("\n🔄 Generating embeddings via OpenAI API and storing in ChromaDB...")
print("⏰ This may take a few minutes depending on the number of chunks...")
print("🌐 Using OpenAI API - embeddings generated automatically by ChromaDB!")

start_time = time.time()
batch_size = 100  # OpenAI can handle larger batches efficiently

for i in range(0, len(total_chunks_to_add), batch_size):
    batch = total_chunks_to_add[i:i+batch_size]
    batch_num = (i // batch_size) + 1
    total_batches = (len(total_chunks_to_add) + batch_size - 1) // batch_size
    
    print(f"\n📦 Processing batch {batch_num}/{total_batches} ({len(batch)} chunks)...")
    
    # Extract texts and metadata
    texts = [chunk["text"] for chunk in batch]
    metadatas = [chunk["metadata"] for chunk in batch]
    
    # Create unique IDs
    ids = [f"{metadatas[j]['category']}_{metadatas[j]['source_file']}_{metadatas[j]['chunk_index']}" 
           for j in range(len(batch))]
    
    # Add to ChromaDB - embeddings are generated automatically by OpenAI
    print(f"   🌐 Calling OpenAI API for embeddings...")
    print(f"   💾 Storing in vector database...")
    try:
        collection.add(
            documents=texts,
            metadatas=metadatas,
            ids=ids
        )
        print(f"   ✅ Batch {batch_num} stored successfully")
    except Exception as e:
        print(f"   ⚠️  Error storing batch {batch_num}: {e}")
        print(f"   ⏳ Retrying after 5 seconds...")
        time.sleep(5)
        collection.add(
            documents=texts,
            metadatas=metadatas,
            ids=ids
        )
        print(f"   ✅ Batch {batch_num} stored successfully (retry)")
    
    # Small delay between batches to avoid rate limiting
    if batch_num < total_batches:
        time.sleep(1)

elapsed_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"✅ VECTOR STORE CREATION COMPLETE!")
print(f"{'='*80}")
print(f"🌐 Embeddings generated via: OpenAI API")
print(f"🤖 Model: {EMBEDDING_MODEL}")
print(f"📐 Dimensions: 1536")
print(f"💰 Cost: $0.02 per 1M tokens")
print(f"🎯 Collection name: {COLLECTION_NAME}")
print(f"📁 Vector store location: {VECTOR_STORE_PATH}")
print(f"💾 Total chunks stored: {len(total_chunks_to_add):,}")
print(f"⏱️  Total time: {elapsed_time:.2f} seconds")
print(f"{'='*80}")


🔄 Generating embeddings via OpenAI API and storing in ChromaDB...
⏰ This may take a few minutes depending on the number of chunks...
🌐 Using OpenAI API - embeddings generated automatically by ChromaDB!

📦 Processing batch 1/4 (100 chunks)...
   🌐 Calling OpenAI API for embeddings...
   💾 Storing in vector database...
   ✅ Batch 1 stored successfully

📦 Processing batch 2/4 (100 chunks)...
   🌐 Calling OpenAI API for embeddings...
   💾 Storing in vector database...
   ✅ Batch 2 stored successfully

📦 Processing batch 3/4 (100 chunks)...
   🌐 Calling OpenAI API for embeddings...
   💾 Storing in vector database...
   ✅ Batch 3 stored successfully

📦 Processing batch 4/4 (12 chunks)...
   🌐 Calling OpenAI API for embeddings...
   💾 Storing in vector database...
   ✅ Batch 4 stored successfully

✅ VECTOR STORE CREATION COMPLETE!
🌐 Embeddings generated via: OpenAI API
🤖 Model: text-embedding-3-small
📐 Dimensions: 1536
💰 Cost: $0.02 per 1M tokens
🎯 Collection name: wwf_knowledge_base
📁 Vecto

## Step 5: Validate Vector Store

In [8]:
print("\n🔍 Validating vector store...")

# Get collection stats
collection_count = collection.count()
print(f"\n✅ Collection '{COLLECTION_NAME}' contains {collection_count:,} chunks")

# Test retrieval for each category
print("\n📊 Testing metadata filtering by category:")
for category in CATEGORIES:
    results = collection.get(
        where={"category": category},
        limit=5
    )
    
    chunk_count = len(results['ids'])
    print(f"\n📂 {category}:")
    print(f"   ✅ Found {chunk_count} chunks (showing first 5)")
    
    if chunk_count > 0:
        # Show sample
        sample_text = results['documents'][0][:200]
        sample_metadata = results['metadatas'][0]
        print(f"   📄 Source: {sample_metadata['source_file']}")
        print(f"   🆔 CourseID: {sample_metadata.get('course_id', 'N/A')}")
        print(f"   📝 Preview: {sample_text}...")

# Test filtering by CourseID
print("\n\n🔍 Testing CourseID filtering:")
for course_id, category in COURSE_ID_TO_CATEGORY.items():
    results = collection.get(
        where={"course_id": course_id},
        limit=3
    )
    print(f"\n🆔 CourseID '{course_id}' ({category}):")
    print(f"   ✅ Found {len(results['ids'])} chunks (showing 3)")

# Test semantic search
print("\n\n🔍 Testing semantic search:")
test_query = "What is sustainable agriculture?"
print(f"Query: '{test_query}'")

# ChromaDB will automatically generate embeddings for the query using OpenAI
search_results = collection.query(
    query_texts=[test_query],
    n_results=3
)

print(f"\n✅ Found {len(search_results['ids'][0])} relevant chunks:")
for idx, (doc, metadata) in enumerate(zip(search_results['documents'][0], search_results['metadatas'][0]), 1):
    print(f"\n   Result {idx}:")
    print(f"   📂 Category: {metadata['category']}")
    print(f"   🆔 CourseID: {metadata.get('course_id', 'N/A')}")
    print(f"   📄 Source: {metadata['source_file']}")
    print(f"   📝 Preview: {doc[:150]}...")

print("\n" + "="*80)
print("✅ VALIDATION COMPLETE - Vector store is ready for use!")
print("="*80)
print("\n📋 Metadata Schema:")
print("   - category: For chatbot filtering")
print("   - course_id: For API/payload filtering (e.g., '001', '002', '003')")
print("   - source_file: Original PDF filename")
print("   - chunk_index: Position in document")
print("   - total_chunks: Total chunks from source")
print("   - doc_type: Document type (pdf)")
print("   - ingestion_date: When chunk was created")


🔍 Validating vector store...

✅ Collection 'wwf_knowledge_base' contains 312 chunks

📊 Testing metadata filtering by category:

📂 circular_economy_and_waste_reduction:
   ✅ Found 5 chunks (showing first 5)
   📄 Source: cewr_1.pdf
   🆔 CourseID: 001
   📝 Preview: --- Page 1 ---
 
PURPOSE OF THIS DOCUMENT ................................ ................................ ................................ ............. 1 
BACKGROUND ..................................

📂 sustainability_strategy_and_compliance:
   ✅ Found 5 chunks (showing first 5)
   📄 Source: ssc_1.pdf
   🆔 CourseID: 002
   📝 Preview: --- Page 1 ---
 
 
 
 
Guidance Note 
Integrating ESG factors  into 
financial models for  
infrastructure investments  
 
© GLOBAL WARMING IMAGES / WWF...

📂 sustainable_agriculture_and_natural_resources:
   ✅ Found 5 chunks (showing first 5)
   📄 Source: agri_nr_1.pdf
   🆔 CourseID: 003
   📝 Preview: --- Page 1 ---
Conservation 
and Sustainable 
Livelihoods
 
in Partnership with Local 
Comm

## Summary and Next Steps

**Vector Store Created Successfully!**

**Location:** `C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\vector_store`

**Collection:** `wwf_knowledge_base`

**Metadata Schema:**
Each chunk includes:
- **`category`**: Category folder name (for chatbot filtering)
- **`course_id`**: CourseID (e.g., '001', '002', '003') for API payload filtering
- **`source_file`**: Original PDF filename
- **`chunk_index`**: Chunk position in the document
- **`total_chunks`**: Total number of chunks from the source
- **`doc_type`**: Document type (pdf)
- **`ingestion_date`**: ISO timestamp of ingestion

**Usage:**
- **Micro-learning API**: Filter by `course_id` from request payload
- **Chatbot RAG**: Filter by `category` for context-aware responses
- **Multi-category search**: Query across all categories or filter by metadata

**Next Steps:**
1. Test micro-module generation (see `02_test_microlearning_generation.ipynb`)
2. Integrate with FastAPI endpoint (filter by `course_id`)
3. Build chatbot with category-based filtering
4. Deploy to production with same vector store